In [ ]:
import sys
from rdkit.Chem import AllChem 
from rdkit import Chem
from rdkit.Chem import PandasTools
import pandas as pd
from pathlib import Path
import os
import py3Dmol
from pathlib import Path
sys.path.append("/Users/bertilledelloye/python-course/Project-Xplosion-Ba4/src")
from coordchem.viz.ligand_data import LIGAND_SMILES
from coordchem.parser import parse_formula 
from coordchem.parser import ParsedComplex

def drawit(m,p=None,confId=-1):
        mb = Chem.MolToMolBlock(m,confId=confId) # Returns SDF file as a string
        if p is None:
            p = py3Dmol.view(width=400,height=400)
        p.removeAllModels()
        p.addModel(mb,'sdf') 
        p.setStyle({'stick':{}})
        p.setBackgroundColor('white')
        p.zoomTo()
        return p.show()

#créer une molecule rdkit pour un ligand en ayant deja le smile:
def ligand_3D(ligand_smile):
      ligand_mol=Chem.MolFromSmiles(ligand_smile)
      if ligand_mol is None:
        raise ValueError("Invalid SMILES string.")
      ligand_mol=Chem.AddHs(ligand_mol)
      status = AllChem.EmbedMolecule(ligand_mol)

      if status != 0:
        raise ValueError("3D embedding failed.")

      AllChem.MMFFOptimizeMolecule(ligand_mol)

      return ligand_mol

#if __name__ == "__main__":
    #thalidomide_smiles = "O=C1c2ccccc2C(=O)N1[C@H]3CCC(=O)NC3=O"

    #thalidomide_3d = ligand_3D(thalidomide_smiles)

    #drawit(thalidomide_3d)
#print(thalidomide_3d.GetNumConformers())
#drawit(thalidomide_3d)

def get_smile_from_mol(molecule):
    parsed=parse_formula(molecule)
    ligand_smiles=[]
    for ligand, count in parsed.ligands.items():
        ligand_smile=LIGAND_SMILES[ligand]
        for _ in range(count):
            ligand_smiles.append(ligand_smile)
    #metal = Chem.MolFromSmiles("[Fe]")
    #ligand = Chem.MolFromSmiles("Cl")
    #mol_smile = Chem.CombineMols(metal, ligand)
    #return mol_smile
    return ligand_smiles

#molecule=input("molecule formula")
#if __name__ == "__main__":
 #   molecule_smiles=get_smile_from_mol(molecule)
  #  for smile in molecule_smiles:
   #     molecule_3D=ligand_3D(smile)
    #    drawit(molecule_3D)
                            
#maintenant on veut associer le metal avec les lignads pour fromer le complexe en 3D:
#pour les ligands monodentates:
def complex_3D(formula):
    parsed = parse_formula(formula)

    rw = Chem.RWMol()#version modifiable  de la molecule

    metal_atom = Chem.Atom(parsed.metal)
    metal_idx = rw.AddAtom(metal_atom)#permet de construire la molecule avec le metal au centre
    coords={}
    coords[metal_idx]=(0.0,0.0,0.0)
    positions = octahedral_positions()
    #puis pour les ligands:
    site_index = 0

    for ligand_symbol, count in parsed.ligands.items():
        ligand_smile = LIGAND_SMILES[ligand_symbol]
        donor_symbol = parsed.donor_atoms[ligand_symbol]

        for _ in range(count):
            ligand_mol = ligand_3D(ligand_smile)
            donor_idx_local = find_donor_atom(ligand_mol, donor_symbol)
            target=positions[site_index] #position ou on met l'atome donneur du ligand
            ligand_conf=ligand_mol.GetConformer()
            donor_position=ligand_conf.GetAtomPosition(donor_idx_local)
            offset = rw.GetNumAtoms()

            for atom in ligand_mol.GetAtoms():
                global_idx = rw.AddAtom(atom)

                old_pos = ligand_conf.GetAtomPosition(atom.GetIdx())

                coords[global_idx] = (
                old_pos.x - donor_position.x + target[0],
                old_pos.y - donor_position.y + target[1],
                old_pos.z - donor_position.z + target[2],)#permet de copier les atomes de chaque ligand puis de le mettre a la bonne place


            for bond in ligand_mol.GetBonds():
                a1 = bond.GetBeginAtomIdx() + offset
                a2 = bond.GetEndAtomIdx() + offset
                rw.AddBond(a1, a2, bond.GetBondType())

            donor_idx_global = donor_idx_local + offset
            rw.AddBond(metal_idx, donor_idx_global, Chem.BondType.DATIVE)
        
            site_index += 1
    mol=rw.GetMol()
    conf = Chem.Conformer(mol.GetNumAtoms())
    for atom_idx,position in coords.items():
        conf.SetAtomPosition(atom_idx,position)
    mol.AddConformer(conf)
    return mol
def octahedral_positions(distance=2.0):
    return [
        ( distance, 0.0, 0.0),
        (-distance, 0.0, 0.0),
        (0.0,  distance, 0.0),
        (0.0, -distance, 0.0),
        (0.0, 0.0,  distance),
        (0.0, 0.0, -distance),
    ]
def find_donor_atom(ligand_mol, donor_symbol):
    for atom in ligand_mol.GetAtoms():
        if atom.GetSymbol() == donor_symbol:
            return atom.GetIdx()

    raise ValueError(f"No donor atom {donor_symbol} found.")


mol=complex_3D("[Fe(CN)6]4-")
print(mol.GetNumConformers())
drawit(mol)

1


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:

"""
coordchem/viz/structure_3d.py
-----------------------------
3D structure construction for coordination complexes.

The module turns a :class:`coordchem.parser.ParsedComplex` into an RDKit
:class:`Mol` object carrying a 3D conformer that can be displayed with
``py3Dmol`` (e.g. inside a Streamlit app).

Public API
~~~~~~~~~~
``build_ligand_3d(smiles)``
    Build an RDKit ``Mol`` for a single ligand with explicit hydrogens
    and a single 3D conformer.

``find_donor_atom(mol, donor_symbol, override=None)``
    Locate the donor atom index inside a ligand ``Mol``.

``geometry_positions(geometry, n, distance=2.0)``
    Return ``n`` coordination-site positions arranged according to
    ``geometry`` ("octahedral", "tetrahedral", "square planar", ...).

``build_complex_3d(parsed, distance=2.0)``
    Assemble the metal + ligands into a single RDKit ``Mol`` with a
    conformer. Suitable for display.

``view_complex_3d(parsed, width=400, height=400)``
    Return a ``py3Dmol.view`` ready to be displayed.

``complex_3d_html(parsed, width=400, height=400)``
    Return a self-contained HTML snippet — convenient for embedding in
    Streamlit through ``st.components.v1.html``.
"""

from __future__ import annotations

from typing import Iterable, Optional, Sequence, Tuple
import math
from rdkit import Chem
from rdkit.Chem import AllChem

from coordchem.parser import ParsedComplex, parse_formula, KNOWN_LIGANDS
from coordchem.viz.ligand_data import LIGAND_DONOR_INDEX_OVERRIDES, LIGAND_SMILES
from coordchem.name2 import parse_name

Position = Tuple[float, float, float]


# ---------------------------------------------------------------------------
# Geometry placeholders
# ---------------------------------------------------------------------------

def octahedral_positions(distance: float = 2.0) -> list[Position]:
    """Return six orthogonal positions on the +/- x, y, z axes."""
    return [
        ( distance,  0.0,       0.0),
        (-distance,  0.0,       0.0),
        ( 0.0,       distance,  0.0),
        ( 0.0,      -distance,  0.0),
        ( 0.0,       0.0,       distance),
        ( 0.0,       0.0,      -distance),
    ]
def octahedral_positions_bidentate(sites):
    return [(sites[0],sites[2]),(sites[1],sites[4]),(sites[3],sites[5])]

def tetrahedral_positions(distance: float = 2.0) -> list[Position]:
    """Return four positions pointing to the corners of a tetrahedron."""
    a = distance / (3 ** 0.5)
    return [
        ( a,  a,  a),
        ( a, -a, -a),
        (-a,  a, -a),
        (-a, -a,  a),
    ]


def square_planar_positions(distance: float = 2.0) -> list[Position]:
    """Return four positions in the xy plane."""
    return [
        ( distance, 0.0, 0.0),
        (-distance, 0.0, 0.0),
        ( 0.0,  distance, 0.0),
        ( 0.0, -distance, 0.0),
    ]


def linear_positions(distance: float = 2.0) -> list[Position]:
    """Return two collinear positions along x."""
    return [( distance, 0.0, 0.0), (-distance, 0.0, 0.0)]


def trigonal_planar_positions(distance: float = 2.0) -> list[Position]:
    """Return three positions in the xy plane, 120° apart."""
    import math
    return [
        (distance * math.cos(math.radians(angle)),
         distance * math.sin(math.radians(angle)),
         0.0)
        for angle in (0, 120, 240)
    ]


def trigonal_bipyramidal_positions(distance: float = 2.0) -> list[Position]:
    """Three equatorial + two axial positions."""
    return trigonal_planar_positions(distance) + [
        (0.0, 0.0,  distance),
        (0.0, 0.0, -distance),
    ]


# Mapping from geometry label (as produced by ``coordchem.geometry``) to
# the corresponding position generator. ``geometry_positions`` does a
# best-effort lookup so that ambiguous labels like
# "trigonal bipyramidal or square pyramidal" still produce something
# reasonable.
_GEOMETRY_BUILDERS = {
    "linear": linear_positions,
    "trigonal planar": trigonal_planar_positions,
    "tetrahedral": tetrahedral_positions,
    "square planar": square_planar_positions,
    "trigonal bipyramidal": trigonal_bipyramidal_positions,
    "octahedral": octahedral_positions,
}


def geometry_positions(
    geometry: str | None,
    n: int,
    distance: float = 2.0,
) -> list[Position]:
    """
    Return ``n`` coordination-site positions matching ``geometry``.

    Falls back to an octahedral arrangement (truncated/extended to ``n``)
    when the geometry label is unknown — useful as a TODO placeholder for
    geometries we have not implemented yet.
    """
    if geometry:
        for label, builder in _GEOMETRY_BUILDERS.items():
            if label in geometry.lower():
                positions = builder(distance)
                if len(positions) >= n:
                    return positions[:n]
                # not enough sites: extend with octahedral ones
                extra = octahedral_positions(distance)[: n - len(positions)]
                return positions + extra

    # Default: octahedral, padded if needed
    base = octahedral_positions(distance)
    if n <= len(base):
        return base[:n]
    # TODO: implement explicit positions for CN > 6
    return base + [(0.0, 0.0, 0.0)] * (n - len(base))


# ---------------------------------------------------------------------------
# Ligand-level helpers
# ---------------------------------------------------------------------------

def build_ligand_3d(smiles: str) -> Chem.Mol:
    """
    Build an RDKit ``Mol`` from ``smiles`` with explicit Hs and a
    single 3D conformer.

    Raises
    ------
    ValueError
        If the SMILES cannot be parsed or a 3D conformer cannot be embedded.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES string: {smiles!r}")

    mol = Chem.AddHs(mol)

    # Ions like [Cl-] or [O-2] have a single atom — nothing to embed.
    if mol.GetNumAtoms() <= 1:
        conf = Chem.Conformer(mol.GetNumAtoms())
        if mol.GetNumAtoms() == 1:
            conf.SetAtomPosition(0, (0.0, 0.0, 0.0))
        mol.AddConformer(conf, assignId=True)
        return mol

    status = AllChem.EmbedMolecule(mol, randomSeed=42)
    if status != 0:
        # Retry with random coordinates as fallback
        status = AllChem.EmbedMolecule(
            mol, randomSeed=42, useRandomCoords=True
        )
        if status != 0:
            raise ValueError(f"3D embedding failed for SMILES: {smiles!r}")

    try:
        AllChem.MMFFOptimizeMolecule(mol)
    except Exception:
        # Optimization is best-effort: keep the embedded geometry
        pass

    return mol


def find_donor_atom2(
    mol: Chem.Mol,
    donor_symbol: str,
    override: Optional[int] = None,
) -> int:
    """
    Return the atom index of the donor inside a ligand ``Mol``.

    ``override`` is the explicit index from
    :data:`coordchem.viz.ligand_data.LIGAND_DONOR_INDEX_OVERRIDES`
    when available. Otherwise the first atom matching ``donor_symbol``
    is returned.

    Multi-donor symbols like ``"N/O"`` (used by EDTA) try each candidate
    in order.

    Raises
    ------
    ValueError
        When no atom with the requested element symbol exists.
    """
    if override is not None:
        if 0 <= override < mol.GetNumAtoms():
            return override

    candidates: Iterable[str]
    if donor_symbol == "?" or not donor_symbol:
        # Unknown: take the first heavy atom
        for atom in mol.GetAtoms():
            if atom.GetSymbol() != "H":
                return atom.GetIdx()
        raise ValueError("Ligand has no non-H atoms")

    candidates = [s.strip() for s in donor_symbol.split("/") if s.strip()]
    for symbol in candidates:
        for atom in mol.GetAtoms():
            if atom.GetSymbol() == symbol:
                return atom.GetIdx()

    raise ValueError(f"No donor atom matching {donor_symbol!r} found in ligand")

#on peut prendre directement les infos données dans KNOWNLIGANDS a partir de .donor_atoms


def find_donor_atom(mol, donor_symbol):
    for atom in mol.GetAtoms():
        if atom.GetSymbol() == donor_symbol:
            return atom.GetIdx()

    raise ValueError(f"No donor atom {donor_symbol} found in ligand")

        
 #function for the disosition of bidentate ligands 


def vec_sub(a, b):
    return (a[0] - b[0], a[1] - b[1], a[2] - b[2])

def vec_add(a, b):
    return (a[0] + b[0], a[1] + b[1], a[2] + b[2])

def vec_scale(v, s):
    return (v[0] * s, v[1] * s, v[2] * s)

def dot(a, b):
    return a[0]*b[0] + a[1]*b[1] + a[2]*b[2]

def cross(a, b):
    return (
        a[1]*b[2] - a[2]*b[1],
        a[2]*b[0] - a[0]*b[2],
        a[0]*b[1] - a[1]*b[0],
    )

def norm(v):
    return (v[0]**2 + v[1]**2 + v[2]**2) ** 0.5

def unit(v):
    n = norm(v)
    if n == 0:
        return (1.0, 0.0, 0.0)
    return (v[0]/n, v[1]/n, v[2]/n)

def rotate_vector(v, axis, angle):
    import math

    axis = unit(axis)
    cos_a = math.cos(angle)
    sin_a = math.sin(angle)

    term1 = vec_scale(v, cos_a)
    term2 = vec_scale(cross(axis, v), sin_a)
    term3 = vec_scale(axis, dot(axis, v) * (1 - cos_a))

    return vec_add(vec_add(term1, term2), term3)

def translate_bidentate_ligand(ligand_mol, donor_indices, target_sites):
    ligand_conf = ligand_mol.GetConformer()

    donor1 = ligand_conf.GetAtomPosition(donor_indices[0])
    donor2 = ligand_conf.GetAtomPosition(donor_indices[1])

    donor1 = (donor1.x, donor1.y, donor1.z)
    donor2 = (donor2.x, donor2.y, donor2.z)

    ligand_mid = (
    (donor1[0] + donor2[0]) / 2,
    (donor1[1] + donor2[1]) / 2,
    (donor1[2] + donor2[2]) / 2,
)

    target1, target2 = target_sites

    target_mid = (
    (target1[0] + target2[0]) / 2,
    (target1[1] + target2[1]) / 2,
    (target1[2] + target2[2]) / 2,
)
    push= 2.0
    outward=unit(target_mid)
    
    target_mid = (
        target_mid[0] + outward[0] * push,
        target_mid[1] + outward[1] * push,
        target_mid[2] + outward[2] * push,
    )

    ligand_axis = unit(vec_sub(donor2, donor1))
    target_axis = unit(vec_sub(target2, target1))

    rotation_axis=cross(ligand_axis, target_axis)
    rotation_axis_norm=norm(rotation_axis)

    if rotation_axis_norm == 0:
        angle = 0.0
        rotation_axis = (1.0, 0.0, 0.0)
    else:
        cos_angle = max(-1.0, min(1.0, dot(ligand_axis, target_axis)))
        angle = math.acos(cos_angle)

    coords = {}
    for atom in ligand_mol.GetAtoms():
        idx = atom.GetIdx()
        old_pos = ligand_conf.GetAtomPosition(idx)
        old_pos = (old_pos.x, old_pos.y, old_pos.z)

        centered = vec_sub(old_pos, ligand_mid)
        rotated = rotate_vector(centered, rotation_axis, angle)
        final_pos = vec_add(rotated, target_mid)

        coords[idx] = final_pos
    return coords
       
# ---------------------------------------------------------------------------
# Complex-level builder
# ---------------------------------------------------------------------------
#à garder ou non car ne change pas grand chose
def ammonia_ligand_positions(ligand_mol, donor_idx, target, nh_distance=1.0, spread=0.8):
    direction = unit(target)

    center = target
    coords = {}
    coords[donor_idx] = center

    if abs(direction[0]) < 0.9:
        ref = (1.0, 0.0, 0.0)
    else:
        ref = (0.0, 1.0, 0.0)

    u = unit(cross(direction, ref))
    v = unit(cross(direction, u))

    h_indices = [
        atom.GetIdx()
        for atom in ligand_mol.GetAtoms()
        if atom.GetSymbol() == "H"
    ]

    import math

    for i, h_idx in enumerate(h_indices[:3]):
        angle = 2 * math.pi * i / 3

        radial = vec_add(
            vec_scale(u, math.cos(angle) * spread),
            vec_scale(v, math.sin(angle) * spread),
        )

        backward = vec_scale(direction, -0.4)

        h_pos = vec_add(center, vec_add(radial, backward))
        coords[h_idx] = h_pos

    for atom in ligand_mol.GetAtoms():
        idx = atom.GetIdx()
        if idx not in coords:
            coords[idx] = center

    return coords


def build_complex_3d(
    parsed: ParsedComplex,
    distance: float = 2.0,
    geometry: str | None = None,
) -> Chem.Mol:

    if geometry is None:
        try:
            from coordchem.geometry import predict_geometry
            geometry = predict_geometry(parsed)
        except Exception:
            geometry = "octahedral"

    cn = parsed.coordination_number or 6
    sites = geometry_positions(geometry, cn, distance=distance)

    rw = Chem.RWMol()
    coords: dict[int, Position] = {}

    metal_atom = Chem.Atom(parsed.metal)
    metal_atom.SetNoImplicit(True)
    metal_idx = rw.AddAtom(metal_atom)
    coords[metal_idx] = (0.0, 0.0, 0.0)

    site_index = 0

    for ligand_symbol, count in parsed.ligands.items():
        smiles = LIGAND_SMILES.get(ligand_symbol)

        if smiles is None:
            for _ in range(count):
                if site_index >= len(sites):
                    break

                placeholder = Chem.Atom("X")
                idx = rw.AddAtom(placeholder)
                coords[idx] = sites[site_index]
                rw.AddBond(metal_idx, idx, Chem.BondType.DATIVE)
                site_index += 1

            continue

        donor_symbol = parsed.donor_atoms.get(ligand_symbol, "?")
        donor_overrides = LIGAND_DONOR_INDEX_OVERRIDES.get(ligand_symbol, ())
        denticity = parsed.ligand_denticity.get(ligand_symbol, 1)

        for _ in range(count):
            if site_index >= len(sites):
                break

            try:
                ligand_mol = build_ligand_3d(smiles)
            except ValueError:
                continue

            if denticity == 2 and len(donor_overrides) >= 2:
                bidentate_index = site_index // 2

                if "octahedral" in geometry.lower() and len(sites) >= 6:
                    site_pairs = octahedral_positions_bidentate(sites)
                else:
                    site_pairs = [
                        (sites[i], sites[i + 1])
                        for i in range(0, len(sites) - 1, 2)
                    ]

                if bidentate_index >= len(site_pairs):
                    break

                target_sites = site_pairs[bidentate_index]
                donor_indices = donor_overrides[:2]

                local_coords = translate_bidentate_ligand(
                    ligand_mol,
                    donor_indices,
                    target_sites,
                )

                offset = rw.GetNumAtoms()

                for atom in ligand_mol.GetAtoms():
                    global_idx = rw.AddAtom(atom)
                    coords[global_idx] = local_coords[atom.GetIdx()]

                for bond in ligand_mol.GetBonds():
                    a1 = bond.GetBeginAtomIdx() + offset
                    a2 = bond.GetEndAtomIdx() + offset
                    rw.AddBond(a1, a2, bond.GetBondType())

                for donor_idx in donor_indices:
                    rw.AddBond(
                        metal_idx,
                        donor_idx + offset,
                        Chem.BondType.DATIVE,
                    )

                site_index += 2
                continue

            primary_override = donor_overrides[0] if donor_overrides else None

            if primary_override is not None and primary_override < ligand_mol.GetNumAtoms():
                donor_idx_local = primary_override
            else:
                try:
                    donor_idx_local = find_donor_atom(ligand_mol, donor_symbol)
                except ValueError:
                    continue
#maintenant c'est le cas monodentate
            target = sites[site_index]

            if ligand_symbol="NH3":
                ammonia_ligand_positions(ligand_mol, donor_idx_local,target)

            elif ligand_mol.GetNumAtoms() == 2:
                local_coords = diatomic_ligand_positions(
                    ligand_mol,
                    donor_idx_local,
                    target,
                    bond_length=1.2,
                )
            else:
                ligand_conf = ligand_mol.GetConformer()
                donor_pos = ligand_conf.GetAtomPosition(donor_idx_local)

                local_coords = {}

                for atom in ligand_mol.GetAtoms():
                    old_pos = ligand_conf.GetAtomPosition(atom.GetIdx())
                    local_coords[atom.GetIdx()] = (
                        old_pos.x - donor_pos.x + target[0],
                        old_pos.y - donor_pos.y + target[1],
                        old_pos.z - donor_pos.z + target[2],
                    )

            offset = rw.GetNumAtoms()

            for atom in ligand_mol.GetAtoms():
                global_idx = rw.AddAtom(atom)
                coords[global_idx] = local_coords[atom.GetIdx()]

            for bond in ligand_mol.GetBonds():
                a1 = bond.GetBeginAtomIdx() + offset
                a2 = bond.GetEndAtomIdx() + offset
                rw.AddBond(a1, a2, bond.GetBondType())

            rw.AddBond(
                metal_idx,
                donor_idx_local + offset,
                Chem.BondType.DATIVE,
            )

            site_index += 1

    mol = rw.GetMol()
    conf = Chem.Conformer(mol.GetNumAtoms())

    for atom_idx, position in coords.items():
        conf.SetAtomPosition(atom_idx, position)
    mol.AddConformer(conf, assignId=True)
    return mol

   

def build_complex_3d_2(
    parsed: ParsedComplex,
    distance: float = 2.0,
    geometry: str | None = None,
) -> Chem.Mol:
    """
    Assemble a 3D RDKit ``Mol`` for the given :class:`ParsedComplex`.

    The metal sits at the origin and ligand donor atoms are placed on
    coordination sites generated by :func:`geometry_positions`.

    Parameters
    ----------
    parsed:
        Parsed coordination complex.
    distance:
        Metal–donor distance in Å.
    geometry:
        Optional override for the geometry label. If ``None``, the
        prediction returned by :func:`coordchem.geometry.predict_geometry`
        is used. Falls back to "octahedral" when unavailable.
    """
    if geometry is None:
        try:
            from coordchem.geometry import predict_geometry
            geometry = predict_geometry(parsed)
        except Exception:
            geometry = "octahedral"

    cn = parsed.coordination_number or 6
    sites = geometry_positions(geometry, cn, distance=distance)

    rw = Chem.RWMol()
    coords: dict[int, Position] = {}

    # Metal at origin
    metal_atom = Chem.Atom(parsed.metal)
    metal_atom.SetNoImplicit(True)
    metal_idx = rw.AddAtom(metal_atom)
    coords[metal_idx] = (0.0, 0.0, 0.0)

    site_index = 0

    for ligand_symbol, count in parsed.ligands.items():
        smiles = LIGAND_SMILES.get(ligand_symbol)
        if smiles is None:
            # Unknown ligand: skip, but record a placeholder atom on the site
            # so the user still sees something where the ligand should be.
            for _ in range(count):
                if site_index >= len(sites):
                    break
                placeholder = Chem.Atom("X")  # dummy
                idx = rw.AddAtom(placeholder)
                coords[idx] = sites[site_index]
                rw.AddBond(metal_idx, idx, Chem.BondType.DATIVE)
                site_index += 1
            continue

        donor_symbol = parsed.donor_atoms.get(ligand_symbol, "?")
        donor_overrides: Sequence[int] = LIGAND_DONOR_INDEX_OVERRIDES.get(
            ligand_symbol, ()
        )

        for _ in range(count):
            if site_index >= len(sites):
                break

            try:
                ligand_mol = build_ligand_3d(smiles)
            except ValueError:
                continue  # skip a ligand that fails to embed

            denticity= parsed.ligand_denticity.get(ligand_symbol,1)
            donor_indices = LIGAND_DONOR_INDEX_OVERRIDES.get(ligand_symbol, ())
            if denticity == 2 and len(donor_indices) >= 2:
                bidentate_index = site_index // 2
                target_sites = octahedral_positions_bidentate(sites)[bidentate_index]

                donor_indices = donor_indices[:2]

                local_coords = translate_bidentate_ligand(
                ligand_mol,donor_indices,target_sites,)

                offset = rw.GetNumAtoms()

                for atom in ligand_mol.GetAtoms():
                    global_idx = rw.AddAtom(atom)
                    coords[global_idx] = local_coords[atom.GetIdx()]

                for bond in ligand_mol.GetBonds():
                    a1 = bond.GetBeginAtomIdx() + offset
                    a2 = bond.GetEndAtomIdx() + offset
                    rw.AddBond(a1, a2, bond.GetBondType())

                for donor_idx in donor_indices:
                    rw.AddBond(metal_idx, donor_idx + offset, Chem.BondType.DATIVE,)

                site_index += 2
                continue
                
            primary_override = (
                donor_overrides[0] if donor_overrides else None
            )
            try:
                donor_idx_local = find_donor_atom(
                    ligand_mol, donor_symbol
                )
            except ValueError:
                continue

            target = sites[site_index]
            if ligand_mol.GetNumAtoms() == 2:
                local_coords = diatomic_ligand_positions(
        ligand_mol,
        donor_idx_local,
        target,
        bond_length=1.2,
    )
            else:
                ligand_conf = ligand_mol.GetConformer()
                donor_pos = ligand_conf.GetAtomPosition(donor_idx_local)

                local_coords = {}

                for atom in ligand_mol.GetAtoms():
                    old_pos = ligand_conf.GetAtomPosition(atom.GetIdx())
                    local_coords[atom.GetIdx()] = (
            old_pos.x - donor_pos.x + target[0],
            old_pos.y - donor_pos.y + target[1],
            old_pos.z - donor_pos.z + target[2],
        )
        offset = rw.GetNumAtoms()
        site_index+=1
            # Copy atoms with translated coordinates so that the donor
            # atom lands on ``target``.
        for atom in ligand_mol.GetAtoms():
            global_idx = rw.AddAtom(atom)
            old_pos = ligand_conf.GetAtomPosition(atom.GetIdx())
            coords[global_idx] = (
                    old_pos.x - donor_pos.x + target[0],
                    old_pos.y - donor_pos.y + target[1],
                    old_pos.z - donor_pos.z + target[2],
                )

            # Copy intra-ligand bonds
        for bond in ligand_mol.GetBonds():
            a1 = bond.GetBeginAtomIdx() + offset
            a2 = bond.GetEndAtomIdx() + offset
            rw.AddBond(a1, a2, bond.GetBondType())

            # Metal–donor dative bond for the primary site
        rw.AddBond(
                metal_idx,
                donor_idx_local + offset,
                Chem.BondType.DATIVE,
            )
            #le probleme ici est que l'index de 1 ne marceh que pour les monodentates
            

            # TODO: properly distribute remaining donors of polydentate
            # ligands across multiple sites. For now we just consume one
            # site per ligand instance and rely on the geometry/2D
            # diagram to communicate the denticity.

    mol = rw.GetMol()
    conf = Chem.Conformer(mol.GetNumAtoms())
    for atom_idx, position in coords.items():
        conf.SetAtomPosition(atom_idx, position)
    mol.AddConformer(conf, assignId=True)

    return mol

#fonctions qui display les ligands diatiomiques pour que le deuxieme atome soit bien positionné
def normalize(v):
    length=(v[0]**2+v[1]**2+v[2]**2)**0.5
    if length==0:
        return(1.0,0.0,0.0)
    return (v[0]/length,v[1]/length,v[2]/length)


def diatomic_ligand_positions(ligand_mol, donor_idx, target, bond_length=1.2):
    direction = normalize(target)
    coords = {}
    for atom in ligand_mol.GetAtoms():
        idx = atom.GetIdx()
        if idx == donor_idx:
            coords[idx] = target
        else:
            coords[idx] = (
                target[0] + direction[0] * bond_length,
                target[1] + direction[1] * bond_length,
                target[2] + direction[2] * bond_length,
            )
    return coords


# ---------------------------------------------------------------------------
# Display helpers
# ---------------------------------------------------------------------------

def _to_parsed(complex_or_formula) -> ParsedComplex:
    """Best-effort coercion to a ``ParsedComplex``."""
    if isinstance(complex_or_formula, ParsedComplex):
        return complex_or_formula
    if isinstance(complex_or_formula, str):
        return parse_complex_input(complex_or_formula)
    parsed = getattr(complex_or_formula, "parsed", None)
    if isinstance(parsed, ParsedComplex):
        return parsed
    raise TypeError(
        "Expected a ParsedComplex, Complex, or formula string"
    )
#test
parsed = parse_complex_input("trisacetylacetonatocobalt(III)")
mol = build_complex_3d(parsed)

print(parsed.ligands)
print(parsed.coordination_number)
print(mol.GetNumAtoms())



#test 



from coordchem.geometry import predict_geometry
geometry = predict_geometry(parsed)
print(geometry)

sites = geometry_positions(geometry, parsed.coordination_number)
print(len(sites))
print(sites)




def view_complex_3d(
    complex_or_formula,
    width: int = 400,
    height: int = 400,
    distance: float = 2.0,
):
    """
    Return a ``py3Dmol.view`` for the complex.

    The caller is responsible for displaying the view (``view.show()``
    in a notebook or ``view._make_html()`` for embedding).
    """
    import py3Dmol  # local import: keep py3Dmol optional

    parsed = _to_parsed(complex_or_formula)
    mol = build_complex_3d(parsed, distance=distance)

    block = Chem.MolToMolBlock(mol, kekulize=False)#pour ne pas forcer la conversion des cycles aromatiques
    view = py3Dmol.view(width=width, height=height)
    view.removeAllModels()
    view.addModel(block, "sdf")
    view.setStyle({}, {"stick": {}, "sphere": {"scale": 0.25}})
    view.setStyle({}, {"stick": {"radius": 0.12}, "sphere": {"scale": 0.18}})

    view.setBackgroundColor("white")
    view.zoomTo()
    return view
#fonction utile pour display la molecule sur un notebook

def complex_3d_html(
    complex_or_formula,
    width: int = 400,
    height: int = 400,
    distance: float = 2.0,
) -> str:
    """Return self-contained HTML for embedding the 3D view."""
    view = view_complex_3d(
        complex_or_formula,
        width=width,
        height=height,
        distance=distance,
    )
    return view._make_html()
#fonction utile pour voir sur une app comme ici streamlit

def parse_complex_input(complex_input):
    if "[" in complex_input and "]" in complex_input:
        return parse_formula(complex_input)
    else:
        return parse_name(complex_input)


view_complex_3d("[Fe(CN)6]4-")

SyntaxError: invalid syntax. Maybe you meant '==' or ':=' instead of '='? (599549661.py, line 513)

In [56]:
view_complex_3d("triiodomercurate(II)")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [87]:
view_complex_3d("hexabromocobalt(III)")


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [142]:
view_complex_3d("trisacetylacetonatocobalt(III)")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [141]:
view_complex_3d("trisacetylacetonatocobalt(III)", distance=3.0)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [139]:
view_complex_3d("tris2,2'bipyridinecobalt(III)", distance=2.5)



3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [104]:
view_complex_3d("hexachlorocobalt(III)")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [118]:
view_complex_3d("hexacyanocobalt(III)")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [148]:
view_complex_3d("diamminesilver(I)")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [149]:
view_complex_3d("diamminedichloroplatinium(II)", distance =1.5)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.